check bronze table

In [0]:
spark.sql("SHOW TABLES IN traveldbs.bronzedbs").show(
    truncate=False
)

+---------+-------------------------+-----------+
|database |tableName                |isTemporary|
+---------+-------------------------+-----------+
|bronzedbs|customers                |false      |
|bronzedbs|delivery_events          |false      |
|bronzedbs|driver_monthly_metrics   |false      |
|bronzedbs|drivers                  |false      |
|bronzedbs|facilities               |false      |
|bronzedbs|fuel_purchases           |false      |
|bronzedbs|loads                    |false      |
|bronzedbs|maintenance_records      |false      |
|bronzedbs|routes                   |false      |
|bronzedbs|safety_incidents         |false      |
|bronzedbs|trailers                 |false      |
|bronzedbs|trips                    |false      |
|bronzedbs|truck_utilization_metrics|false      |
|bronzedbs|trucks                   |false      |
+---------+-------------------------+-----------+



Append

In [0]:
from pyspark.sql import functions as F

# =========================================================
# APPEND OPERATION - BRONZE LAYER
# =========================================================

bronze_table = "traveldbs.bronzedbs.delivery_events"

# Read new/incoming CSV file
new_data = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/traveldbs/bronzedbs/transportationdbs_volume/delivery_events.csv")
)

# Append new records into existing Bronze Delta table
new_data.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(bronze_table)

print("APPEND completed successfully")

APPEND completed successfully


Silver - Clean + OVERWRITE

In [0]:
bronze_tables = [
    row.tableName
    for row in spark.sql("SHOW TABLES IN traveldbs.bronzedbs").collect()
]

for table_name in bronze_tables:

    df = spark.table(f"traveldbs.bronzedbs.{table_name}")

    # Cleaning
    silver_df = (
        df.dropDuplicates()
          .na.drop(how="all")
    )

    # OVERWRITE Silver table
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"traveldbs.silverdbs.{table_name}")

    print(f"Silver created: {table_name}")

Silver created: customers
Silver created: delivery_events
Silver created: driver_monthly_metrics
Silver created: drivers
Silver created: facilities
Silver created: fuel_purchases
Silver created: loads
Silver created: maintenance_records
Silver created: routes
Silver created: safety_incidents
Silver created: trailers
Silver created: trips
Silver created: truck_utilization_metrics
Silver created: trucks


Silver UPDATE + DELETE

In [0]:
from delta.tables import DeltaTable

bronze_df = spark.table("traveldbs.bronzedbs.delivery_events")

silver_df = bronze_df.dropDuplicates()

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("traveldbs.silverdbs.delivery_events")

silver_table = DeltaTable.forName(
    spark,
    "traveldbs.silverdbs.delivery_events"
)

# UPDATE
silver_table.update(
    condition="event_id = 'EV10000010'",
    set={"event_type": "'Delivery'"}
)

# DELETE
silver_table.delete("event_id IS NULL")

print("Silver table created and UPDATE + DELETE completed")

Silver table created and UPDATE + DELETE completed


Merger operation

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "traveldbs.silverdbs.delivery_events"
)

# Create source data for MERGE
merge_source = spark.table(
    "traveldbs.silverdbs.delivery_events"
).filter("event_id = 'EV10000010'")

# MERGE
silver_table.alias("target").merge(
    merge_source.alias("source"),
    "target.event_id = source.event_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("MERGE completed successfully")

MERGE completed successfully
